# JPEG Compression and Decompression Demo

This notebook demonstrates the core concepts of JPEG spatial compression:
1. **DCT (Discrete Cosine Transform)** - Transform spatial domain to frequency domain
2. **Quantization** - Reduce precision of frequency coefficients
3. **Entropy Coding** - Huffman coding for compression
4. **Inverse Operations** - Reconstruct the original image

We'll process an image in 8x8 blocks, just like real JPEG compression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import heapq
from collections import defaultdict, Counter
import pickle
import os

# Set up matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## Step 1: Load the Famous Lena Image

We'll use the classic Lena image, a standard test image in image processing. This provides realistic image content to better demonstrate JPEG compression effects on natural images.

In [ ]:
def load_lena_image():
    """Load the Lena grayscale BMP image"""
    try:
        # Try to load the downloaded image first
        img = Image.open('lena_gray.bmp')
        img_array = np.array(img)
        
        # Convert to grayscale if needed
        if len(img_array.shape) == 3:
            # Convert RGB to grayscale using luminance formula
            img_array = np.dot(img_array[...,:3], [0.2989, 0.5870, 0.1140])
        
        return img_array.astype(np.uint8)
        
    except FileNotFoundError:
        print("Lena image not found locally. Downloading...")
        import urllib.request
        
        # Download the image
        url = 'https://eeweb.engineering.nyu.edu/~yao/EL5123/image/lena_gray.bmp'
        urllib.request.urlretrieve(url, 'lena_gray.bmp')
        
        # Load the downloaded image
        img = Image.open('lena_gray.bmp')
        img_array = np.array(img)
        
        if len(img_array.shape) == 3:
            img_array = np.dot(img_array[...,:3], [0.2989, 0.5870, 0.1140])
        
        return img_array.astype(np.uint8)

# Load Lena image
full_lena = load_lena_image()

# For demonstration purposes, we'll use a smaller portion (256x256) to speed up processing
# You can use the full image by changing this
crop_size = 256
start_y = (full_lena.shape[0] - crop_size) // 2
start_x = (full_lena.shape[1] - crop_size) // 2
original_img = full_lena[start_y:start_y+crop_size, start_x:start_x+crop_size]

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(full_lena, cmap='gray')
plt.title(f'Full Lena Image\n{full_lena.shape[0]}x{full_lena.shape[1]}')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(original_img, cmap='gray')
plt.title(f'Cropped for Demo\n{original_img.shape[0]}x{original_img.shape[1]}')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.hist(original_img.flatten(), bins=50, alpha=0.7, color='skyblue')
plt.title('Pixel Intensity Histogram')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Full image shape: {full_lena.shape}")
print(f"Demo image shape: {original_img.shape}")
print(f"Image dtype: {original_img.dtype}")
print(f"Pixel value range: {original_img.min()} to {original_img.max()}")
print(f"Mean pixel value: {original_img.mean():.1f}")
print(f"Standard deviation: {original_img.std():.1f}")

## Step 2: JPEG Quantization Table

The quantization table determines how much information is lost in each frequency band. Lower values preserve more detail, higher values compress more aggressively.

In [ ]:
# Standard JPEG luminance quantization table (quality ~ 50%)
QUANTIZATION_TABLE = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61],
    [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77],
    [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103, 99]
], dtype=np.float32)

plt.figure(figsize=(8, 6))
plt.imshow(QUANTIZATION_TABLE, cmap='hot')
plt.title('JPEG Quantization Table\n(Higher values = more compression)')
plt.colorbar()

# Add text annotations
for i in range(8):
    for j in range(8):
        plt.text(j, i, f'{int(QUANTIZATION_TABLE[i,j])}', 
                ha='center', va='center', color='white', fontsize=10)

plt.xlabel('Frequency (horizontal)')
plt.ylabel('Frequency (vertical)')
plt.show()

print("Note: Top-left corner represents DC (average) component")
print("Bottom-right corner represents highest frequency components")

## Step 3: DCT (Discrete Cosine Transform)

DCT converts spatial domain data to frequency domain. It concentrates most energy in low-frequency components (top-left of the 8x8 block).

In [ ]:
def dct_2d(block):
    """2D DCT transform for 8x8 block"""
    # Shift to center around 0 (subtract 128 for 8-bit images)
    centered_block = block.astype(np.float32) - 128
    
    # Create DCT basis functions
    N = 8
    dct_result = np.zeros((N, N), dtype=np.float32)
    
    for u in range(N):
        for v in range(N):
            cu = 1/np.sqrt(2) if u == 0 else 1
            cv = 1/np.sqrt(2) if v == 0 else 1
            
            sum_val = 0
            for x in range(N):
                for y in range(N):
                    sum_val += centered_block[x, y] * \
                              np.cos((2*x + 1) * u * np.pi / (2*N)) * \
                              np.cos((2*y + 1) * v * np.pi / (2*N))
            
            dct_result[u, v] = (2/N) * cu * cv * sum_val
    
    return dct_result

def idct_2d(dct_block):
    """2D Inverse DCT transform for 8x8 block"""
    N = 8
    spatial_result = np.zeros((N, N), dtype=np.float32)
    
    for x in range(N):
        for y in range(N):
            sum_val = 0
            for u in range(N):
                for v in range(N):
                    cu = 1/np.sqrt(2) if u == 0 else 1
                    cv = 1/np.sqrt(2) if v == 0 else 1
                    
                    sum_val += cu * cv * dct_block[u, v] * \
                              np.cos((2*x + 1) * u * np.pi / (2*N)) * \
                              np.cos((2*y + 1) * v * np.pi / (2*N))
            
            spatial_result[x, y] = (2/N) * sum_val
    
    # Add back the 128 offset and clip to valid range
    spatial_result += 128
    return np.clip(spatial_result, 0, 255)

# Test DCT on first 8x8 block
test_block = original_img[:8, :8].astype(np.float32)
dct_coeffs = dct_2d(test_block)

# Visualize the transformation
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(test_block, cmap='gray')
plt.title('Original 8x8 Block')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(dct_coeffs, cmap='RdBu_r')
plt.title('DCT Coefficients')
plt.colorbar()

# Test reconstruction
reconstructed = idct_2d(dct_coeffs)
plt.subplot(1, 3, 3)
plt.imshow(reconstructed, cmap='gray')
plt.title('Reconstructed Block\n(should match original)')
plt.colorbar()

plt.tight_layout()
plt.show()

print(f"Reconstruction error (MSE): {np.mean((test_block - reconstructed)**2):.6f}")
print(f"Max absolute error: {np.max(np.abs(test_block - reconstructed)):.6f}")

## Step 4: Quantization

Quantization reduces the precision of DCT coefficients by dividing by the quantization table and rounding. This is where most information loss occurs.

In [ ]:
def quantize(dct_block, q_table):
    """Quantize DCT coefficients"""
    return np.round(dct_block / q_table).astype(np.int16)

def dequantize(quantized_block, q_table):
    """Dequantize coefficients"""
    return quantized_block * q_table

# Apply quantization to our test block
quantized_coeffs = quantize(dct_coeffs, QUANTIZATION_TABLE)
dequantized_coeffs = dequantize(quantized_coeffs, QUANTIZATION_TABLE)

# Visualize quantization effects
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(dct_coeffs, cmap='RdBu_r', vmin=-100, vmax=100)
plt.title('Original DCT Coefficients')
plt.colorbar()

plt.subplot(1, 4, 2)
plt.imshow(quantized_coeffs, cmap='RdBu_r')
plt.title('Quantized Coefficients\n(integers only)')
plt.colorbar()

# Add text annotations for quantized values
for i in range(8):
    for j in range(8):
        if abs(quantized_coeffs[i,j]) > 0:
            plt.text(j, i, f'{int(quantized_coeffs[i,j])}', 
                    ha='center', va='center', color='yellow', fontsize=8, weight='bold')

plt.subplot(1, 4, 3)
plt.imshow(dequantized_coeffs, cmap='RdBu_r', vmin=-100, vmax=100)
plt.title('Dequantized Coefficients\n(with quantization error)')
plt.colorbar()

# Reconstruct from quantized coefficients
lossy_reconstruction = idct_2d(dequantized_coeffs)
plt.subplot(1, 4, 4)
plt.imshow(lossy_reconstruction, cmap='gray')
plt.title('Lossy Reconstruction')
plt.colorbar()

plt.tight_layout()
plt.show()

# Show difference
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(test_block, cmap='gray')
plt.title('Original Block')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(lossy_reconstruction, cmap='gray')
plt.title('Compressed/Decompressed Block')
plt.colorbar()

plt.subplot(1, 3, 3)
difference = test_block - lossy_reconstruction
plt.imshow(difference, cmap='RdBu_r')
plt.title('Difference (Compression Error)')
plt.colorbar()

plt.tight_layout()
plt.show()

print(f"Quantization introduced MSE: {np.mean(difference**2):.2f}")
print(f"Non-zero quantized coefficients: {np.count_nonzero(quantized_coeffs)}/64")
print(f"Compression ratio (rough): {64/np.count_nonzero(quantized_coeffs):.1f}:1")

## Step 5: Huffman Coding for Entropy Compression

Huffman coding assigns shorter bit codes to more frequent values, achieving lossless compression of the quantized coefficients.

In [ ]:
class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None
    
    def __lt__(self, other):
        return self.freq < other.freq

class HuffmanCoder:
    def __init__(self):
        self.codes = {}
        self.root = None
    
    def build_tree(self, frequency_dict):
        """Build Huffman tree from frequency dictionary"""
        heap = [HuffmanNode(char, freq) for char, freq in frequency_dict.items()]
        heapq.heapify(heap)
        
        while len(heap) > 1:
            left = heapq.heappop(heap)
            right = heapq.heappop(heap)
            
            merged = HuffmanNode(None, left.freq + right.freq)
            merged.left = left
            merged.right = right
            
            heapq.heappush(heap, merged)
        
        self.root = heap[0] if heap else None
        self._generate_codes(self.root, "")
    
    def _generate_codes(self, node, code):
        """Generate Huffman codes recursively"""
        if node is not None:
            if node.char is not None:  # Leaf node
                self.codes[node.char] = code if code else "0"  # Handle single symbol case
            else:
                self._generate_codes(node.left, code + "0")
                self._generate_codes(node.right, code + "1")
    
    def encode(self, data):
        """Encode data using Huffman codes"""
        encoded = []
        for symbol in data:
            encoded.append(self.codes[symbol])
        return ''.join(encoded)
    
    def decode(self, encoded_string):
        """Decode Huffman encoded string"""
        if not self.root:
            return []
        
        decoded = []
        current_node = self.root
        
        for bit in encoded_string:
            if bit == '0':
                current_node = current_node.left
            else:
                current_node = current_node.right
            
            if current_node.char is not None:  # Leaf node
                decoded.append(current_node.char)
                current_node = self.root
        
        return decoded

def zigzag_order():
    """Generate zigzag order for 8x8 block (JPEG standard)"""
    zigzag = []
    for i in range(8):
        if i % 2 == 0:  # Even diagonals go up-right
            for j in range(i + 1):
                if i - j < 8 and j < 8:
                    zigzag.append((i - j, j))
        else:  # Odd diagonals go down-left
            for j in range(i + 1):
                if j < 8 and i - j < 8:
                    zigzag.append((j, i - j))
    
    # Continue with the remaining diagonals
    for i in range(1, 8):
        if (i + 7) % 2 == 0:  # Even diagonals go up-right
            for j in range(8 - i):
                if 7 - j < 8 and i + j < 8:
                    zigzag.append((7 - j, i + j))
        else:  # Odd diagonals go down-left
            for j in range(8 - i):
                if i + j < 8 and 7 - j < 8:
                    zigzag.append((i + j, 7 - j))
    
    return zigzag[:64]  # Ensure we have exactly 64 elements

# Generate zigzag pattern
zigzag_indices = zigzag_order()

# Convert quantized coefficients to 1D array in zigzag order
zigzag_data = [quantized_coeffs[i, j] for i, j in zigzag_indices]

# Create frequency table
freq_table = Counter(zigzag_data)

# Build Huffman tree and encode
huffman = HuffmanCoder()
huffman.build_tree(freq_table)

encoded_bitstring = huffman.encode(zigzag_data)

# Visualize zigzag pattern
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
zigzag_visual = np.zeros((8, 8))
for idx, (i, j) in enumerate(zigzag_indices):
    zigzag_visual[i, j] = idx
plt.imshow(zigzag_visual, cmap='viridis')
plt.title('Zigzag Scan Order')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.bar(range(len(freq_table)), list(freq_table.values()))
plt.xlabel('Coefficient Value')
plt.ylabel('Frequency')
plt.title('Coefficient Frequency Distribution')
plt.xticks(range(len(freq_table)), list(freq_table.keys()), rotation=45)

plt.subplot(1, 3, 3)
code_lengths = [len(code) for code in huffman.codes.values()]
plt.hist(code_lengths, bins=range(1, max(code_lengths)+2), alpha=0.7)
plt.xlabel('Code Length (bits)')
plt.ylabel('Number of Symbols')
plt.title('Huffman Code Length Distribution')

plt.tight_layout()
plt.show()

print("Huffman Codes:")
for symbol, code in sorted(huffman.codes.items()):
    print(f"  {symbol:3d}: {code} ({len(code)} bits)")

print(f"\nOriginal data: {zigzag_data}")
print(f"\nCompression Statistics:")
print(f"  Original bits (8 bits per coefficient): {64 * 8} bits")
print(f"  Huffman encoded bits: {len(encoded_bitstring)} bits")
print(f"  Huffman compression ratio: {(64 * 8) / len(encoded_bitstring):.2f}:1")

# Test decoding
decoded_data = huffman.decode(encoded_bitstring)
print(f"\nDecoding successful: {decoded_data == zigzag_data}")

## Step 6: Complete JPEG Encoder/Decoder for Full Image

Now let's implement the complete pipeline for processing the entire image in 8x8 blocks.

In [ ]:
def jpeg_encode_image(image, q_table):
    """Encode entire image using JPEG compression"""
    height, width = image.shape
    
    # Pad image to multiple of 8
    pad_h = (8 - height % 8) % 8
    pad_w = (8 - width % 8) % 8
    
    if pad_h > 0 or pad_w > 0:
        padded_image = np.pad(image, ((0, pad_h), (0, pad_w)), mode='edge')
    else:
        padded_image = image
    
    padded_height, padded_width = padded_image.shape
    
    # Process in 8x8 blocks
    all_coefficients = []
    zigzag_indices = zigzag_order()
    
    for i in range(0, padded_height, 8):
        for j in range(0, padded_width, 8):
            block = padded_image[i:i+8, j:j+8].astype(np.float32)
            
            # DCT transform
            dct_block = dct_2d(block)
            
            # Quantize
            quantized_block = quantize(dct_block, q_table)
            
            # Convert to zigzag order
            zigzag_coeffs = [quantized_block[i, j] for i, j in zigzag_indices]
            all_coefficients.extend(zigzag_coeffs)
    
    # Build frequency table for entire image
    freq_table = Counter(all_coefficients)
    
    # Build Huffman tree
    huffman_coder = HuffmanCoder()
    huffman_coder.build_tree(freq_table)
    
    # Encode coefficients
    encoded_bitstring = huffman_coder.encode(all_coefficients)
    
    return {
        'encoded_data': encoded_bitstring,
        'huffman_codes': huffman_coder.codes,
        'original_shape': image.shape,
        'padded_shape': padded_image.shape,
        'quantization_table': q_table,
        'zigzag_indices': zigzag_indices
    }

def jpeg_decode_image(encoded_info):
    """Decode JPEG compressed image"""
    # Reconstruct Huffman decoder
    huffman_coder = HuffmanCoder()
    huffman_coder.codes = encoded_info['huffman_codes']
    
    # Build reverse lookup for decoding
    reverse_codes = {v: k for k, v in huffman_coder.codes.items()}
    
    # Decode bitstring
    decoded_coeffs = []
    current_code = ""
    
    for bit in encoded_info['encoded_data']:
        current_code += bit
        if current_code in reverse_codes:
            decoded_coeffs.append(reverse_codes[current_code])
            current_code = ""
    
    # Reconstruct image
    padded_height, padded_width = encoded_info['padded_shape']
    reconstructed_image = np.zeros((padded_height, padded_width), dtype=np.float32)
    
    zigzag_indices = encoded_info['zigzag_indices']
    q_table = encoded_info['quantization_table']
    
    block_idx = 0
    for i in range(0, padded_height, 8):
        for j in range(0, padded_width, 8):
            # Extract 64 coefficients for this block
            block_coeffs = decoded_coeffs[block_idx*64:(block_idx+1)*64]
            
            # Convert from zigzag order to 8x8 block
            quantized_block = np.zeros((8, 8), dtype=np.int16)
            for idx, coeff in enumerate(block_coeffs):
                row, col = zigzag_indices[idx]
                quantized_block[row, col] = coeff
            
            # Dequantize
            dequantized_block = dequantize(quantized_block, q_table)
            
            # Inverse DCT
            spatial_block = idct_2d(dequantized_block)
            
            # Place back in image
            reconstructed_image[i:i+8, j:j+8] = spatial_block
            block_idx += 1
    
    # Remove padding
    orig_height, orig_width = encoded_info['original_shape']
    final_image = reconstructed_image[:orig_height, :orig_width]
    
    return np.clip(final_image, 0, 255).astype(np.uint8)

# Encode the entire image
print("Encoding image...")
encoded_image = jpeg_encode_image(original_img, QUANTIZATION_TABLE)

# Save encoded data to binary file (using relative path)
with open('compressed_image.bin', 'wb') as f:
    pickle.dump(encoded_image, f)

print(f"Compressed data saved to compressed_image.bin")
print(f"File size: {os.path.getsize('compressed_image.bin')} bytes")

# Calculate compression statistics
original_bits = original_img.shape[0] * original_img.shape[1] * 8
compressed_bits = len(encoded_image['encoded_data'])
compression_ratio = original_bits / compressed_bits

print(f"\nCompression Statistics:")
print(f"  Original size: {original_bits} bits ({original_bits//8} bytes)")
print(f"  Compressed size: {compressed_bits} bits ({compressed_bits//8} bytes)")
print(f"  Compression ratio: {compression_ratio:.2f}:1")
print(f"  Space savings: {(1 - compressed_bits/original_bits)*100:.1f}%")

## Step 7: Decode from Binary File and Reconstruct Image

Now we'll read the binary file and perform the complete inverse operation to reconstruct the image.

In [ ]:
# Read compressed data from binary file
print("Loading compressed data from file...")
with open('compressed_image.bin', 'rb') as f:
    loaded_encoded_image = pickle.load(f)

# Decode the image
print("Decoding image...")
reconstructed_img = jpeg_decode_image(loaded_encoded_image)

# Display results
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(original_img, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(reconstructed_img, cmap='gray')
plt.title('Reconstructed Image\n(After Compression/Decompression)')
plt.axis('off')

plt.subplot(1, 3, 3)
difference = original_img.astype(np.float32) - reconstructed_img.astype(np.float32)
plt.imshow(difference, cmap='RdBu_r')
plt.title('Difference\n(Compression Artifacts)')
plt.colorbar()

plt.tight_layout()
plt.show()

# Calculate quality metrics
mse = np.mean((original_img.astype(np.float32) - reconstructed_img.astype(np.float32))**2)
psnr = 10 * np.log10(255**2 / mse) if mse > 0 else float('inf')
max_error = np.max(np.abs(difference))

print(f"\nQuality Metrics:")
print(f"  Mean Squared Error (MSE): {mse:.2f}")
print(f"  Peak Signal-to-Noise Ratio (PSNR): {psnr:.2f} dB")
print(f"  Maximum Absolute Error: {max_error:.1f}")

print(f"\nDecoding successful: Image reconstructed from binary file!")
print(f"Final image shape: {reconstructed_img.shape}")
print(f"Final image dtype: {reconstructed_img.dtype}")

## Step 8: Visualize the Complete Pipeline

Let's create a comprehensive visualization showing all steps of the JPEG pipeline.

In [ ]:
# Demonstrate the complete pipeline on a single block
demo_block = original_img[16:24, 16:24].astype(np.float32)

# Forward pipeline
demo_dct = dct_2d(demo_block)
demo_quantized = quantize(demo_dct, QUANTIZATION_TABLE)
demo_zigzag = [demo_quantized[i, j] for i, j in zigzag_indices]

# Reverse pipeline 
demo_dequantized = dequantize(demo_quantized, QUANTIZATION_TABLE)
demo_reconstructed = idct_2d(demo_dequantized)

# Create comprehensive visualization
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Complete JPEG Compression Pipeline', fontsize=16, fontweight='bold')

# Row 1: Forward Transform
im1 = axes[0, 0].imshow(demo_block, cmap='gray')
axes[0, 0].set_title('1. Original 8×8 Block\n(Spatial Domain)')
plt.colorbar(im1, ax=axes[0, 0], shrink=0.6)

im2 = axes[0, 1].imshow(demo_dct, cmap='RdBu_r')
axes[0, 1].set_title('2. DCT Coefficients\n(Frequency Domain)')
plt.colorbar(im2, ax=axes[0, 1], shrink=0.6)

im3 = axes[0, 2].imshow(demo_quantized, cmap='RdBu_r')
axes[0, 2].set_title('3. Quantized Coefficients\n(Lossy Step)')
plt.colorbar(im3, ax=axes[0, 2], shrink=0.6)
# Add coefficient values
for i in range(8):
    for j in range(8):
        if abs(demo_quantized[i,j]) > 0:
            axes[0, 2].text(j, i, f'{int(demo_quantized[i,j])}', 
                           ha='center', va='center', color='yellow', fontsize=8, weight='bold')

axes[0, 3].plot(demo_zigzag, 'bo-', markersize=4)
axes[0, 3].set_title('4. Zigzag Sequence\n(Ready for Huffman)')
axes[0, 3].set_xlabel('Position in Zigzag Order')
axes[0, 3].set_ylabel('Coefficient Value')
axes[0, 3].grid(True, alpha=0.3)

# Row 2: Quantization Table and Huffman Info
im4 = axes[1, 0].imshow(QUANTIZATION_TABLE, cmap='hot')
axes[1, 0].set_title('Quantization Table\n(Higher = More Compression)')
plt.colorbar(im4, ax=axes[1, 0], shrink=0.6)

# Show coefficient frequency
coeff_freq = Counter(demo_zigzag)
coeffs, freqs = zip(*sorted(coeff_freq.items()))
axes[1, 1].bar(range(len(coeffs)), freqs)
axes[1, 1].set_title('Coefficient Frequency\n(For Huffman Coding)')
axes[1, 1].set_xlabel('Coefficient Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_xticks(range(len(coeffs)))
axes[1, 1].set_xticklabels(coeffs)

# Show compression effect in frequency domain
freq_comparison = np.abs(demo_dct) - np.abs(demo_dequantized)
im5 = axes[1, 2].imshow(freq_comparison, cmap='Reds')
axes[1, 2].set_title('Lost Information\n(|Original| - |Quantized|)')
plt.colorbar(im5, ax=axes[1, 2], shrink=0.6)

# Show energy concentration
energy_original = demo_dct ** 2
energy_quantized = demo_dequantized ** 2
axes[1, 3].plot(np.cumsum(energy_original.flatten())/np.sum(energy_original), 'b-', label='Original DCT')
axes[1, 3].plot(np.cumsum(energy_quantized.flatten())/np.sum(energy_quantized), 'r--', label='Quantized DCT')
axes[1, 3].set_title('Cumulative Energy\n(Shows Information Loss)')
axes[1, 3].set_xlabel('Coefficient Index')
axes[1, 3].set_ylabel('Cumulative Energy Fraction')
axes[1, 3].legend()
axes[1, 3].grid(True, alpha=0.3)

# Row 3: Reverse Transform
im6 = axes[2, 0].imshow(demo_dequantized, cmap='RdBu_r')
axes[2, 0].set_title('5. Dequantized Coefficients\n(Frequency Domain)')
plt.colorbar(im6, ax=axes[2, 0], shrink=0.6)

im7 = axes[2, 1].imshow(demo_reconstructed, cmap='gray')
axes[2, 1].set_title('6. Reconstructed Block\n(Spatial Domain)')
plt.colorbar(im7, ax=axes[2, 1], shrink=0.6)

error = demo_block - demo_reconstructed
im8 = axes[2, 2].imshow(error, cmap='RdBu_r')
axes[2, 2].set_title('7. Reconstruction Error\n(Artifacts)')
plt.colorbar(im8, ax=axes[2, 2], shrink=0.6)

# Show pixel-wise comparison
axes[2, 3].plot(demo_block.flatten(), 'b-', alpha=0.7, label='Original')
axes[2, 3].plot(demo_reconstructed.flatten(), 'r--', alpha=0.7, label='Reconstructed')
axes[2, 3].set_title('Pixel-wise Comparison')
axes[2, 3].set_xlabel('Pixel Index')
axes[2, 3].set_ylabel('Pixel Value')
axes[2, 3].legend()
axes[2, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("JPEG COMPRESSION PIPELINE SUMMARY")
print("="*80)
print("\n📥 ENCODING STEPS:")
print("   1. Divide image into 8×8 blocks")
print("   2. Apply DCT to convert to frequency domain")
print("   3. Quantize coefficients (LOSSY step)")
print("   4. Arrange in zigzag order")
print("   5. Apply Huffman coding (lossless compression)")
print("   6. Save bitstream to binary file")
print("\n📤 DECODING STEPS:")
print("   1. Load bitstream from binary file")
print("   2. Decode Huffman codes")
print("   3. Rearrange from zigzag to 8×8 blocks")
print("   4. Dequantize coefficients")
print("   5. Apply inverse DCT to convert back to spatial domain")
print("   6. Combine 8×8 blocks to reconstruct full image")
print("\n🎯 KEY LEARNING POINTS:")
print("   • DCT concentrates energy in low frequencies (top-left)")
print("   • Quantization causes most information loss")
print("   • Huffman coding provides lossless compression")
print("   • Zigzag order groups similar coefficients together")
print("   • Higher quantization = more compression but lower quality")
print("\n" + "="*80)

## Summary

This notebook demonstrates the complete JPEG compression pipeline:

### 🔄 **The Three Core Steps of Spatial Compression:**

1. **🌊 Transformation (DCT)**: Converts spatial data to frequency domain, concentrating energy in low frequencies
2. **🎯 Quantization**: Reduces precision of frequency coefficients (where most compression occurs)
3. **📦 Entropy Coding**: Uses Huffman coding for lossless compression of quantized values

### 🔄 **The Exact Inverse Steps for Reconstruction:**

1. **📦 Entropy Decoding**: Huffman decode to recover quantized coefficients  
2. **🎯 Dequantization**: Multiply by quantization table (cannot recover lost precision)
3. **🌊 Inverse Transformation**: Apply inverse DCT to return to spatial domain

### 📊 **Key Educational Insights:**

- **Frequency Domain**: DCT reveals that most image energy is in low frequencies
- **Lossy vs Lossless**: Quantization is lossy, Huffman coding is lossless
- **Trade-offs**: Higher quantization = better compression but more artifacts
- **Block Artifacts**: 8×8 processing can create visible block boundaries
- **Zigzag Order**: Arranges coefficients from low to high frequency for better compression

The binary file demonstrates how the compressed bitstream can be stored and later decoded to reconstruct the image with controlled quality loss.